In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json


In [2]:
if platform.system() == 'Windows':
    os.environ['PYSPARK_PYTHON'] = sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = SparkSession \
    .builder \
    .appName("Data with Nikk the Greek Spark Session") \
    .master("local[4]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

In [11]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze" 
}

In [12]:
@F.udf(returnType="STRING")
def get_properties(url):
   json_request = requests.get(url).json()
   return json.dumps(json_request["result"]["properties"])

In [13]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("properties", get_properties(F.col("url")))
    
bronze_instance = StarWarsBronze(spark, **options)


In [14]:
bronze_instance.load().transform().write(mode="overwrite").execute("people")

In [15]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-01-11 13:23:...|           Boba Fett| 22|https://www.swapi...|{"height": "183",...|
|2025-01-11 13:23:...|               IG-88| 23|https://www.swapi...|{"height": "200",...|
|2025-01-11 13:23:...|               Bossk| 24|https://www.swapi...|{"height": "190",...|
|2025-01-11 13:23:...|    Lando Calrissian| 25|https://www.swapi...|{"height": "177",...|
|2025-01-11 13:23:...|               Lobot| 26|https://www.swapi...|{"height": "175",...|
|2025-01-11 13:23:...|              Ackbar| 27|https://www.swapi...|{"height": "180",...|
|2025-01-11 13:23:...|          Mon Mothma| 28|https://www.swapi...|{"height": "150",...|
|2025-01-11 13:23:...|        Arvel Crynyd| 29|https://www.swapi...|{"height": "unkno..

# 2 Silver

In [16]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [17]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [18]:
class StarWarsSilver(silver.Silver):    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        if table == "people":
            sdf = self.transf_people(sdf)
        return sdf

    def transf_people(self, sdf: DataFrame) -> DataFrame:
        sdf = (
            sdf.withColumn("height", sdf.properties.height)
            .withColumn("mass", sdf.properties.mass)
            .withColumn("gender", sdf.properties.gender)
            .drop("url","properties")
        )
        for i in range(0,35):
            sdf = sdf.withColumn(f"col{str(i)}", F.lit(str(i)))
        return sdf
    
silver_instance = StarWarsSilver(spark, **options)

In [19]:
silver_instance.load().transform().write(mode="overwrite", merge_schema=True).execute("people")

In [20]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 82
+--------------------------+--------------------------+---------------------+---+-------+-------+-------------+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|height |mass   |gender       |col0|col1|col2|col3|col4|col5|col6|col7|col8|col9|col10|col11|col12|col13|col14|col15|col16|col17|col18|col19|col20|col21|col22|col23|col24|col25|col26|col27|col28|col29|col30|col31|col32|col33|col34|
+--------------------------+--------------------------+---------------------+---+-------+-------+-------------+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|2025-01-11 13:24:27.422025|2025-01-11 13:23:37.990

# 3 Optimize Silver

In [25]:
#Set layout for liquid columns and optimize undependend of write operation
(
    silver_instance.tblproperties(clusterby = ["gender"])
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

In [29]:
#You can also do this with the default class
(
    silver.Silver(spark, **options)
    .tblproperties(clusterby = ["gender"])
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

In [28]:
#run above commands also together with the write
(
    silver_instance.load()
    .transform()
    .write(mode="overwrite", merge_schema=True)
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

In [21]:
#run also a FULL Optimize when liquid clustering and a vacuum Lite with Delta 3.3 and above
(
    silver_instance.tblproperties(clusterby = ["gender"])
    .optimize(optimize=True, optimize_full=True, vacuum=True, vacuum_lite=True)
    .execute("people")
)

In [ ]:
# Run analyze and exclude cols you dont want an analyze to run. Especially long strings.
# Does not work with spark_catalog in Spark. 
(
    silver_instance.tblproperties(clusterby = ["gender"])
    .optimize(analyze=True, excl_cols = ["url"])
    .execute("people")
)

# Review in Delta History and details

In [22]:
from delta.tables import *

q = f"DESCRIBE HISTORY {CATALOG}.silver.people"
hist = spark.sql(q)
hist.show(truncate=False)

q = f"DESCRIBE DETAIL {CATALOG}.silver.people"
det = spark.sql(q)
det.show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                           |job |notebook|clusterId|readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                                                                                                                                                                               

# 4 Clean Up

In [34]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")

DataFrame[]